# Treinamento do SegFormer (estágio 10)

Executa o protocolo de treino do SegFormer (ViT) sobre a mesma divisão espacial k-fold e as mesmas estatísticas de normalização do estágio 08, para a **fonte escolhida manualmente** (`ground_truth.chosen_source`, registrada no estágio 04). O protocolo é idêntico ao do estágio 09 (`src/trainer.py`) — mesma perda, otimizador, scheduler, sementes e métricas —; apenas a arquitetura muda. Os artefatos por dobra (pesos, métricas e histórico) são persistidos sob `MyDrive/tcc/models/segformer/<fonte>/` e `MyDrive/tcc/artifacts/{metrics,runs}/segformer/<fonte>/`.


## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.


In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc-lab/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")


## Dependências pinadas

Garante as versões fixadas em `requirements-runtime.txt`, incluindo o `transformers` usado pelo SegFormer, com o mesmo conjunto nas duas plataformas. Quando uma biblioteca com extensão compilada já importada (numpy, scipy, torch, entre outras) diverge da versão pinada, o kernel é reiniciado automaticamente para evitar inconsistência de ABI; nesse caso, o notebook é reexecutado a partir da primeira célula.


In [ ]:
# Garante as versões pinadas e reinicia o kernel quando a ABI carregada divergir.
from src.setup import install_runtime_requirements

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
install_runtime_requirements(requirements)


## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`, incluindo as pastas de modelos, métricas e histórico de execução. Também exibe a variante do SegFormer definida em `model.segformer_variant`.


In [ ]:
# Resolve o contexto de runtime: plataforma, armazenamento canônico e configuração.
from src.data.patch_generation import manifest_path
from src.data.source_decision import chosen_source
from src.setup import initialize

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
source_name = chosen_source()
print(f"Plataforma: {platform}")
print(f"Fonte de treinamento (decisão manual): {source_name}")
print(f"Variante do SegFormer: {config['model']['segformer_variant']}")
print(f"Manifesto: {manifest_path(storage_paths, source_name)}")
print(f"Modelos: {storage_paths['models']}")
print(f"Métricas: {storage_paths['artifacts_metrics']}")


## Reprodutibilidade e exceção documentada de determinismo

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch (cudnn determinístico, benchmark desligado e algoritmos determinísticos) e registra as versões dos pacotes principais do ambiente. O upsampling bilinear do SegFormer não possui backward determinístico no CUDA segundo o PyTorch; por isso esta etapa usa `warn_only=True`, que mantém a checagem ativa mas emite aviso em vez de erro apenas para essas operações. Seeds, cuDNN, ordem dos lotes, pesos pré-treinados e as demais operações permanecem determinísticos, e o `run.meta.json` registra a configuração efetiva.


In [ ]:
# Fixa sementes e flags determinísticas; warn_only cobre o upsampling bilinear do SegFormer.
from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config, warn_only=True)
log_environment()


## Dependência dos estágios 06, 07 e 08 da fonte escolhida

Verifica que o manifesto de patches da fonte escolhida (estágio 06) está disponível e vigente, que a divisão espacial k-fold (estágio 07) preenche a coluna `fold` e tem metadata vigente (schema, fingerprint, k e semente) e que as estatísticas de normalização (estágio 08) estão atuais.


In [ ]:
# Verifica as dependências dos estágios 06 (manifesto), 07 (split) e 08 (estatísticas) da fonte.
from src.data.eda import normalization_stats_is_current, normalization_stats_path
from src.data.patch_generation import (
    load_manifest,
    manifest_has_folds,
    manifest_is_current,
    manifest_path,
)
from src.data.spatial_split import fold_count, split_is_current
from src.utils import check_dependencies

manifest_file = manifest_path(storage_paths, source_name)
check_dependencies(
    {
        "Manifesto (estágio 06)": manifest_file,
        "Estatísticas de normalização (estágio 08)": normalization_stats_path(
            storage_paths, source_name
        ),
    }
)
manifest = load_manifest(storage_paths, source_name)
if not manifest_has_folds(manifest):
    raise ValueError("Coluna fold ausente/incompleta; execute o estágio 07 antes.")
if not split_is_current(storage_paths, source_name):
    raise ValueError("Divisão espacial desatualizada; execute o estágio 07 antes.")
if not normalization_stats_is_current(storage_paths, source_name):
    raise ValueError("Estatísticas de normalização desatualizadas; execute o estágio 08 antes.")
if not manifest_is_current(storage_paths, source_name):
    raise ValueError("Manifesto desatualizado frente às entradas; execute o estágio 06 antes.")
print(f"Patches: {len(manifest)} | Dobras: {fold_count()}")


## Modelo SegFormer

Constrói o SegFormer a partir de `model.segformer_variant` (padrão `nvidia/mit-b1`), adaptando a primeira convolução para as quatro bandas do Sentinel-2 (B2, B3, B4, B8): os pesos RGB pré-treinados são preservados e o quarto canal é inicializado com a média dos três. O checkpoint é de backbone (ImageNet), então a cabeça de decodificação é inicializada aleatoriamente e ajustada no treino. A saída é um canal de logit binário reamostrado para a resolução da entrada.


In [ ]:
# Constrói o SegFormer a partir de config.yaml e exibe o número de parâmetros.
from src.models.segformer import build_segformer

model = build_segformer()
n_params = sum(parameter.numel() for parameter in model.parameters())
print(f"SegFormer criado: {n_params:,} parâmetros")
print(f"Variante: {config['model']['segformer_variant']}")
print(f"Bandas de entrada: {model.in_channels}")


## Preparação local dos patches

Garante, no Kaggle, que todos os patches do manifesto da fonte tenham cópia no cache local antes do treino, evitando requisições à Drive API durante o carregamento por lote; no-op no Colab/local.


In [ ]:
# Garante cópias locais dos patches (Kaggle); no-op no Colab/local.
from src.data.dataset import ensure_patches_local

local_files = ensure_patches_local(storage_paths, manifest)
print(f"Arquivos preparados localmente: {local_files}")


## Treinamento por dobra

Executa o protocolo de treino único (`src.trainer.train_fold`) para cada dobra da validação espacial da fonte escolhida, construindo um SegFormer novo por dobra com a semente da dobra (`seed + fold`) fixada antes da inicialização dos pesos. Artefatos de uma execução anterior só são reutilizados quando o fingerprint do dataset, o fingerprint do split e o hash da configuração coincidem; caso contrário, a dobra é retreinada automaticamente. A célula exige GPU: interrompe a execução quando o acelerador não está ativo, evitando um treino inviável em CPU.


In [ ]:
# Executa o protocolo de treino único por dobra da fonte escolhida (reutiliza artefatos).
import torch

from src.data.spatial_split import fold_count
from src.models.segformer import build_segformer
from src.trainer import train_fold
from src.utils import set_all_seeds

# Exige GPU: o treino em CPU é inviável para 512x512 por 100 épocas.
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU indisponível: ative o acelerador de GPU (T4/P100) na plataforma antes de treinar."
    )
device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
print(f"Dispositivo: {device} | {torch.cuda.get_device_name(0)} | {props.total_memory / 1e9:.1f} GB")
results = []
for fold in range(fold_count()):
    # Fixa a semente antes de inicializar os pesos, tornando cada dobra reproduzível.
    set_all_seeds(ctx.seed + fold)
    model = build_segformer()
    result = train_fold(model, "segformer", storage_paths, fold, device, source_name=source_name)
    results.append(result)


## Resumo do treinamento e artefatos

Exibe o resumo por dobra (melhor época e IoU de validação), persiste o metadata de execução (configuração do modelo, fonte, fingerprints e versões do ambiente) em `MyDrive/tcc/artifacts/runs/segformer/<fonte>/`.


In [ ]:
# Exibe o resumo do treinamento e persiste o metadata de execução da fonte.
from src.trainer import save_run_metadata
from src.utils import print_summary

meta_file = save_run_metadata(storage_paths, "segformer", source_name)
summary = {"Modelo": "segformer", "Fonte de treinamento": source_name, "Dobras treinadas": len(results)}
for result in results:
    tag = " (reutilizada)" if result.skipped else ""
    summary[f"Dobra {result.fold} — IoU val"] = (
        f"{result.val_metrics['iou']:.4f} (época {result.best_epoch}){tag}"
    )
summary["Pesos por dobra"] = str(results[0].weights_path.parent)
summary["Metadata de execução"] = str(meta_file)
print_summary(summary, "10")
